# 第 2 章：让业务知识可以按语义被找到

**本章目标**：理解文本嵌入（Embedding）与相似度；认识文档和查询的编码契约；比较精确扫描、近似索引与增量更新。代码只使用本地文件和内存向量，不调用模型应用程序编程接口（API, Application Programming Interface），也不连接数据库。

假设用户说“退款后的营收”，而知识卡标题是“净销售额”。只按字面匹配可能漏掉它；嵌入模型把两种表达映射到向量空间，让语义相近的文本有机会靠近。这是检索增强生成（RAG, Retrieval-Augmented Generation）的候选召回手段，并不保证召回的规则必然正确。

## 2.1 一个可手算的向量例子

余弦相似度比较两个向量的方向：

$$\operatorname{cos}(q,d)=\frac{\sum_i q_i d_i}{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}.$$

设问题向量为 <code>q=(1,0)</code>，卡片 A 为 <code>(0.8,0.6)</code>，卡片 B 为 <code>(0,1)</code>。A 与问题的余弦相似度是 0.8，B 是 0；因此 A 排在前面。生产中的向量有上千维，含义由模型学习，不能像这个二维玩具例子那样逐轴解释。

In [1]:
from math import sqrt

def cosine(a, b):
    if len(a) != len(b):
        raise ValueError("向量维度不一致")
    norm_a = sqrt(sum(x * x for x in a))
    norm_b = sqrt(sum(x * x for x in b))
    if norm_a == 0 or norm_b == 0:
        raise ValueError("零向量无法计算余弦相似度")
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)

question = (1.0, 0.0)
candidates = {"A": (0.8, 0.6), "B": (0.0, 1.0)}
scores = {name: cosine(question, vector) for name, vector in candidates.items()}
print(scores)
assert scores["A"] > scores["B"]
print("距离越小越近：", {name: round(1 - score, 3) for name, score in scores.items()})

{'A': 0.8, 'B': 0.0}
距离越小越近： {'A': 0.2, 'B': 1.0}


项目在 pgvector 中使用 <code>embedding &lt;=&gt; query_vector</code>，这是**余弦距离**，数值越小越近。距离不是答案正确概率；它只负责初步排序。精确术语、订单编号和字段名可能更适合词法或结构化检索，下一章会做对照。

> **为什么有查询向量和文档向量两种模式？**
>
> 本项目用 Qwen 的 <code>text_type="document"</code> 为入库知识编码，用 <code>text_type="query"</code> 为用户问题编码，查询侧还带检索任务指令。两侧必须使用兼容的模型与维度。把旧模型的文档向量和新模型的问题向量直接比较，距离失去原来的解释基础。

## 2.2 项目的嵌入文本到底是什么

<code>KnowledgeDoc.embedding_text</code> 把类型、对象、别名和正文组成待嵌入文本；<code>content_hash</code> 对这段文本计算摘要。这里直接使用项目类，但不调用 <code>load_knowledge()</code>：后者会连接数据库，把实时字段加入表卡。

In [2]:
from pathlib import Path
import json
import sys

root = Path.cwd()
if root.name == "tutorial":
    root = root.parent
root = root.resolve()
sys.path.insert(0, str(root / "src"))
from nl2sql.knowledge import KnowledgeDoc

raw = json.loads((root / "db" / "knowledge.json").read_text(encoding="utf-8"))
item = next(card for card in raw if card["id"] == "metric.net_sales")
doc = KnowledgeDoc(
    doc_id=item["id"], kind=item["kind"], object_ref=item["object_ref"],
    content=item["content"], aliases=item["aliases"],
    related_tables=item["related_tables"],
)
print(doc.embedding_text)
print("文档标识符：", doc.doc_id)
print("内容摘要前 12 位：", doc.content_hash[:12])

类型：metric
对象：net_sales
别名：净销售额、净收入、退款后销售额
说明：净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。
文档标识符： metric.net_sales
内容摘要前 12 位： 490c3f90268c


稳定的文档标识符（ID, Identifier）用于引用；内容哈希用于判断“需要重新嵌入吗”。这两个值不能混用：运营改了退款定义，仍是同一张业务卡，ID 不变，但待嵌入文本和哈希应改变。项目的索引器还把模型名纳入变更判断，并在生成全部向量后才执行数据库写入。

In [3]:
from dataclasses import replace

same = replace(doc)
edited = replace(doc, aliases=[*doc.aliases, "退款后收入"])
print("未修改时哈希相同：", same.content_hash == doc.content_hash)
print("新增别名后 ID 相同：", edited.doc_id == doc.doc_id)
print("新增别名后哈希改变：", edited.content_hash != doc.content_hash)
assert same.content_hash == doc.content_hash
assert edited.doc_id == doc.doc_id and edited.content_hash != doc.content_hash

未修改时哈希相同： True
新增别名后 ID 相同： True
新增别名后哈希改变： True


## 2.3 从向量到可维护的索引

| 检索方式 | 适用规模 | 优势 | 代价 |
|---|---|---|---|
| 精确最近邻扫描 | 小知识库或评估基线 | 结果完整，可作为近似索引的对照 | 文档变多时查询成本上升 |
| 分层可导航小世界（HNSW, Hierarchical Navigable Small World）索引 | 较大向量集合、低延迟查询 | 常有较好的速度与召回折中 | 建索引较慢、占内存；可能漏候选 |
| 倒排文件扁平索引（IVFFlat, Inverted File Flat） | 有足够数据并能调参 | 构建和内存开销通常较低 | 列表数与探测数影响召回 |

pgvector 默认支持精确最近邻；近似最近邻（ANN, Approximate Nearest Neighbor）索引以可能漏掉部分邻居为代价加速查询。项目的模式文件创建了 1024 维向量列和 HNSW 余弦索引，但只有十余张知识卡时，没有理由从这个样本推断大规模性能。上线前应拿代表性问题对比**精确扫描与近似索引的召回和延迟**。

In [4]:
schema = (root / "db" / "schema.sql").read_text(encoding="utf-8")
embedding_source = (root / "src" / "nl2sql" / "embedding.py").read_text(encoding="utf-8")
indexer_source = (root / "src" / "nl2sql" / "indexer.py").read_text(encoding="utf-8")
checks = {
    "1024 维列": "vector(1024)" in schema,
    "HNSW 余弦索引": "USING hnsw (embedding vector_cosine_ops)" in schema,
    "文档/查询分开编码": 'text_type not in {"query", "document"}' in embedding_source,
    "哈希与模型联合比较": "(doc.content_hash, settings.qwen_embed_model)" in indexer_source,
}
print(checks)
assert all(checks.values())

{'1024 维列': True, 'HNSW 余弦索引': True, '文档/查询分开编码': True, '哈希与模型联合比较': True}


## 2.4 离线模拟增量更新

项目 <code>index_knowledge()</code> 的核心判断是：源卡的 <code>(content_hash, embed_model)</code> 与索引记录一致，就无需再次请求向量。下面只在内存中模拟。实际运行索引器会访问 Qwen 并写数据库；本 Notebook 不会触发它。

In [5]:
model = "qwen3.7-text-embedding"
indexed_manifest = {doc.doc_id: (doc.content_hash, model)}
current_docs = [doc]
def changed_ids(source_docs, manifest, model_name):
    return [
        item.doc_id for item in source_docs
        if manifest.get(item.doc_id) != (item.content_hash, model_name)
    ]

print("重复索引需更新：", changed_ids(current_docs, indexed_manifest, model))
print("修改别名后需更新：", changed_ids([edited], indexed_manifest, model))
print("升级模型后需更新：", changed_ids(current_docs, indexed_manifest, "new-model"))
assert changed_ids(current_docs, indexed_manifest, model) == []
assert changed_ids([edited], indexed_manifest, model) == [doc.doc_id]

重复索引需更新： []
修改别名后需更新： ['metric.net_sales']
升级模型后需更新： ['metric.net_sales']


模型升级还需要检查维度、数据库列、近似索引和查询端模型。若新模型输出维度不等于 <code>vector(1024)</code>，简单修改配置会失败。更稳妥的迁移顺序是“建新索引版本 → 离线核对召回 → 切换读流量 → 留出回滚窗口”。不要在已有知识卡上直接混用两种模型的向量。

索引发布还要处理删除：源文件里已经没有的 ID 必须从可检索集合中移除。项目索引器在事务中完成更新和删除，但生产系统还应记录发布时间、版本号、失败数与回滚点。

## 练习

**基础练习**：把示例向量 A 改为 <code>(0.6,0.8)</code>，重新算余弦相似度与距离；再给 <code>edited</code> 添加第二个别名，观察哈希是否再变。**检查标准**：能解释“相似度越大 / 距离越小越近”；ID 保持不变，文本改变导致哈希改变。

**进阶练习**：假设向量模型从 1024 维升级到 1536 维，写出新列或新表、重建索引、离线对照、切换与回滚的五步计划。**检查标准**：每步有可验证条件；明确查询向量与文档向量属于同一模型版本；用必要知识召回率和延迟共同验收，而非只比较维度。

**故障题**：用户输入精确的 <code>payments.paid_at</code> 时，仅凭语义向量相近就能保证找到支付时间字段吗？不能；需词法或结构化字段检索，并实际核对候选。

## 继续阅读

- [pgvector 官方索引说明](https://github.com/pgvector/pgvector)
- [阿里云百炼文本嵌入接口](https://help.aliyun.com/zh/model-studio/text-embedding-synchronous-api/)
- [项目索引器](../src/nl2sql/indexer.py)